# Copertura degli intervalli sul regime randomizzato - otto mondi

**Autosufficiente.** Carica solo questo file (`File -> Carica notebook`) e premi **Esegui tutto**. Non c'e' niente da copiare, da incollare o da eseguire prima.

**Una cosa sola da fare:** quando compare il popup di Google Drive, autorizza l'accesso con lo stesso account su cui sta la cartella `MMM_copertura`. E' l'unico momento in cui serve una persona: dopo quel click il notebook va avanti da solo.

**Runtime:** GPU T4. Il file se la porta dietro nelle proprie impostazioni; se Colab non la applica, il pre-volo si ferma alla prima riga e scrive cosa cliccare.

**Cosa fa:** rigenera gli otto mondi dello studio osservativo (semi 42, 101-107) con la spesa **randomizzata** invece che osservativa, li fitta con configurazione identica (96 nodi, 2000 draw tenuti, 4 catene) e misura per ognuno intervallo esatto, PIT, dettaglio per canale e draw del posterior.

**Quanto dura:** circa dieci minuti a mondo su T4, ottanta-centotrenta minuti in tutto. Gira non presidiato; ogni mondo stampa il proprio tempo e il totale trascorso.

**Se Colab cade:** riapri il notebook e premi di nuovo Esegui tutto. Riprende dal mondo successivo all'ultimo registrato, perche' le righe vengono scritte su Drive una per volta, subito dopo il fit.

**Dove scrive:** nella cartella `MMM_copertura` di Drive, accanto al CSV osservativo che non viene mai toccato: `..._randomizzato.csv` (una riga per mondo), `..._randomizzato_canali.csv` (una riga per mondo x canale) e `..._randomizzato_draw/` (un `.csv.gz` per mondo).

**Come si legge:** la regola e' fissata **prima** di eseguire, in `PREREGISTRAZIONE_copertura_randomizzata.md`. L'ultima cella la applica da sola e stampa il ramo: 1, 2, 3, 4 o 5. Il ramo si calcola, non si sceglie a risultati visti.

**Alla fine:** scarica da Drive tutti e tre gli output e riporta in chat la riga `RAMO PRE-REGISTRATO`, per intera e come prima riga della risposta.

In [ ]:
# ============================================================================
# CELLA 1 - PRE-VOLO (sei controlli) E SETUP: repo, patch del generatore, runner
# ============================================================================
# Questa cella non fitta niente: verifica che il run di ottanta minuti abbia
# senso PRIMA di cominciarlo, e prepara il codice. Se anche uno solo dei sei
# controlli fallisce si ferma qui e dice cosa cliccare: un run non presidiato
# che parte con la GPU spenta o con i semi sbagliati e' un run buttato.
import base64, csv, json, os, shutil, subprocess, sys

A = chr(39)          # apostrofo: le stringhe di questo repo restano ASCII
SPUNTA = chr(10003)  # segno di spunta nel riepilogo
CROCE = chr(10007)   # crocetta nel riepilogo

# ============================================================================
# COSTANTI - tutto quello che si puo' voler cambiare sta qui e solo qui
# ============================================================================
REPO = '/content/mmm-bayesiano-recruiting'
URL_REPO = 'https://github.com/Giacomod2001/mmm-bayesiano-recruiting'
CARTELLA_DRIVE = '/content/drive/MyDrive/MMM_copertura'
NOME_CSV_OSSERVATIVO = 'copertura_risultati.csv'
SEMI_MONDO = [42, 101, 102, 103, 104, 105, 106, 107]
KEEP = 2000
VERSIONE_MERIDIAN = '1.8.0'

# La configurazione che questo run avra', da confrontare con quella scritta nel
# CSV osservativo. n_knots=96 viene da KNOTS_PER_QUARTER=12 dentro il runner;
# gli altri sono i suoi default, tranne keep che passiamo noi sulla riga di
# comando. Se anche uno solo differisce, il confronto fra i due studi non
# misurerebbe il regime di spesa ma la differenza di configurazione.
ATTESI = {'n_knots': 96.0, 'keep': float(KEEP), 'n_chains': 4.0, 'adapt': 500.0,
          'burnin': 500.0, 'seed_mcmc': 42.0, 'revenue_per_kpi': 40.0}

CSV_OSSERVATIVO = os.path.join(CARTELLA_DRIVE, NOME_CSV_OSSERVATIVO)
CSV_RANDOMIZZATO = (CSV_OSSERVATIVO[:-4] + '_randomizzato.csv'
                    if CSV_OSSERVATIVO.lower().endswith('.csv')
                    else CSV_OSSERVATIVO + '_randomizzato.csv')
GEN = os.path.join(REPO, 'genera_dati_simulati.py')
RUNNER = os.path.join(REPO, 'studio_copertura_runner.py')

# ============================================================================
# PRE-VOLO - sei controlli, nell'ordine in cui conviene farli
# ============================================================================
# L'ordine non e' casuale. Il popup di Drive e' al secondo posto perche' e'
# l'unico momento in cui serve una persona: arriva entro pochi secondi
# dall'avvio, cosi' dopo averlo autorizzato ci si puo' allontanare. Meridian e'
# per ultimo perche' installarlo dura qualche minuto: tutto cio' che puo'
# fallire subito fallisce prima, senza far aspettare.
#
# Un controllo che non si puo' eseguire (perche' dipende da uno fallito prima)
# conta come fallito: non passato e' non passato.
_ESITI = []
_RIMEDI = []
RIGHE_OSSERVATIVE = []


def _registra(numero, titolo, ok, dettaglio, rimedio=''):
    _ESITI.append((numero, titolo, bool(ok), dettaglio))
    if not ok:
        _RIMEDI.append(str(numero) + '. ' + titolo + '\n   ' + dettaglio +
                       ('\n   COSA FARE: ' + rimedio if rimedio else ''))
    print(('   [ok] ' if ok else '   [NO] ') + dettaglio)
    return bool(ok)


print('PRE-VOLO')
print('=' * 72)

# --- 1. la GPU ---------------------------------------------------------------
print(' 1. GPU')
_nome_gpu = ''
try:
    _r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                        capture_output=True, text=True, timeout=120)
    if _r.returncode == 0 and _r.stdout.strip():
        _nome_gpu = _r.stdout.strip().splitlines()[0].strip()
except (OSError, subprocess.SubprocessError):
    _nome_gpu = ''
if not _nome_gpu:
    _ok1 = _registra(1, 'GPU', False, 'nessuna GPU collegata a questo runtime',
                     'Runtime -> Cambia tipo di runtime -> Acceleratore hardware: '
                     'GPU (T4) -> Salva, poi Esegui tutto di nuovo. Su CPU un solo '
                     'mondo richiede ore invece di dieci minuti: otto sono '
                     'impraticabili.')
elif 'T4' in _nome_gpu.upper():
    _ok1 = _registra(1, 'GPU', True, _nome_gpu)
else:
    _ok1 = _registra(1, 'GPU', True, _nome_gpu + ' (non e' + A + ' una T4: va bene '
                     'lo stesso, ma i tempi saranno diversi dai ~10 min a mondo)')

# --- 2. Drive ----------------------------------------------------------------
# Unico momento interattivo di tutto il notebook: autorizza il popup.
print(' 2. Google Drive (autorizza il popup: e' + A + ' l' + A + 'unica cosa da fare)')
_ok2 = False
try:
    from google.colab import drive as _drive
except ImportError:
    _ok2 = _registra(2, 'Drive', False,
                     'non sei su Colab (google.colab non importabile)',
                     'Questo notebook e' + A + ' fatto per Google Colab: caricalo su '
                     'colab.research.google.com.')
else:
    try:
        _drive.mount('/content/drive')
        _montato = os.path.isdir('/content/drive/MyDrive')
    except Exception as _e:
        _montato = False
        _errore_mount = str(_e)[:120]
    else:
        _errore_mount = ''
    if not _montato:
        _ok2 = _registra(2, 'Drive', False,
                         'Drive non montato' + (': ' + _errore_mount if _errore_mount else ''),
                         'Riesegui la cella e clicca "Connetti a Google Drive", poi '
                         'scegli l' + A + 'account e accetta tutte le caselle.')
    elif not os.path.isdir(CARTELLA_DRIVE):
        _ok2 = _registra(2, 'Drive', False,
                         'Drive montato ma la cartella ' + CARTELLA_DRIVE + ' non esiste',
                         'E' + A + ' la cartella dove lo studio osservativo ha scritto i '
                         'suoi risultati. O hai autorizzato un account Drive diverso da '
                         'quello di allora, oppure la cartella e' + A + ' stata spostata o '
                         'rinominata: rimettila in MyDrive/MMM_copertura.')
    else:
        _ok2 = _registra(2, 'Drive', True, CARTELLA_DRIVE + ' raggiungibile')

# --- 3. il CSV dello studio osservativo --------------------------------------
print(' 3. CSV osservativo')
if not _ok2:
    _ok3 = _registra(3, 'CSV osservativo', False,
                     'non eseguito: dipende dal controllo 2 (Drive)')
elif not os.path.exists(CSV_OSSERVATIVO):
    _ok3 = _registra(3, 'CSV osservativo', False,
                     'non trovato: ' + CSV_OSSERVATIVO,
                     'Serve il CSV dello studio osservativo gia' + A + ' fatto: e' + A +
                     ' da li' + A + ' che si leggono semi e configurazione da replicare. '
                     'Ricaricalo in ' + CARTELLA_DRIVE + ' col nome ' +
                     NOME_CSV_OSSERVATIVO + ', oppure cambia NOME_CSV_OSSERVATIVO '
                     'nelle costanti qui sopra.')
else:
    with open(CSV_OSSERVATIVO, newline='', encoding='utf-8') as _f:
        RIGHE_OSSERVATIVE = [r for r in csv.DictReader(_f) if r.get('esito') == 'ok']
    if not RIGHE_OSSERVATIVE:
        _ok3 = _registra(3, 'CSV osservativo', False,
                         'trovato ma senza righe con esito=ok',
                         'Il file c' + A + 'e' + A + ' ma nessun mondo osservativo e' + A +
                         ' andato a buon fine: non c' + A + 'e' + A + ' niente con cui '
                         'confrontare. Riportalo in chat.')
    else:
        _ok3 = _registra(3, 'CSV osservativo', True,
                         str(len(RIGHE_OSSERVATIVE)) + ' mondi con esito=ok in ' +
                         NOME_CSV_OSSERVATIVO)

# --- 4. i semi ---------------------------------------------------------------
print(' 4. semi dei mondi')
if not _ok3:
    _ok4 = _registra(4, 'semi', False, 'non eseguito: dipende dal controllo 3 (CSV)')
else:
    _semi_oss = sorted(int(r['seed_mondo']) for r in RIGHE_OSSERVATIVE)
    print('      osservativo: ' + str(_semi_oss))
    print('      questo run : ' + str(sorted(SEMI_MONDO)))
    if len(_semi_oss) != len(set(_semi_oss)):
        _ok4 = _registra(4, 'semi', False, 'semi duplicati nel CSV osservativo',
                         'Lo stesso mondo compare due volte con esito=ok: il CSV e' + A +
                         ' stato scritto due volte. Riportalo in chat prima di procedere.')
    elif _semi_oss != sorted(SEMI_MONDO):
        _ok4 = _registra(4, 'semi', False, 'NON coincidono',
                         'Il confronto sarebbe fra due insiemi di mondi diversi, e non '
                         'misurerebbe piu' + A + ' il regime di spesa. Riportalo in chat: '
                         'va deciso a mano quale dei due insiemi e' + A + ' quello giusto.')
    else:
        _ok4 = _registra(4, 'semi', True, 'gli stessi otto, nessun duplicato')

# --- 5. i sette campi di configurazione --------------------------------------
print(' 5. configurazione (sette campi)')
if not _ok3:
    _ok5 = _registra(5, 'configurazione', False,
                     'non eseguito: dipende dal controllo 3 (CSV)')
else:
    _problemi = []
    for _campo, _atteso in sorted(ATTESI.items()):
        _valori = sorted({r.get(_campo, '') for r in RIGHE_OSSERVATIVE})
        if len(_valori) != 1:
            _problemi.append(_campo + ': nel CSV osservativo ha PIU' + A + ' valori ' +
                             str(_valori) + ' (i mondi osservativi non sono nemmeno '
                             'confrontabili fra loro)')
            print('      ' + _campo.ljust(16) + ' ' + str(_valori) + '   <-- NON UNICO')
            continue
        try:
            _uguale = abs(float(_valori[0]) - _atteso) < 1e-9
        except (TypeError, ValueError):
            _uguale = str(_valori[0]) == str(_atteso)
        print('      ' + _campo.ljust(16) + ' ' + str(_valori[0]).ljust(10) +
              ' atteso ' + str(_atteso) + ('   ok' if _uguale else '   <-- DIVERSO'))
        if not _uguale:
            _problemi.append(_campo + ': osservativo ' + str(_valori[0]) +
                             ', questo run ' + str(_atteso))
    if _problemi:
        _ok5 = _registra(5, 'configurazione', False,
                         str(len(_problemi)) + ' campo/i diverso/i: ' + '; '.join(_problemi),
                         'Il confronto sarebbe nullo: la differenza fra i due studi non '
                         'sarebbe il regime di spesa ma la configurazione. Riportalo in '
                         'chat senza lanciare.')
    else:
        _ok5 = _registra(5, 'configurazione', True, 'identica su tutti e sette i campi')

# --- 6. Meridian esattamente 1.8.0 -------------------------------------------
# Le versioni successive cambiano il backend (float32 invece di float64): il fit
# girerebbe lo stesso ma non sarebbe piu' lo stesso fit dello studio osservativo.
print(' 6. Meridian ' + VERSIONE_MERIDIAN + ' esatta')
_CODICE_VERSIONE = (
    'import sys\n'
    'try:\n'
    '    import importlib.metadata as _md\n'
    '    d = _md.version("google-meridian")\n'
    'except Exception:\n'
    '    d = "assente"\n'
    'try:\n'
    '    import meridian\n'
    '    i = meridian.__version__\n'
    'except Exception as e:\n'
    '    i = "NON IMPORTABILE (" + type(e).__name__ + ": " + str(e)[:100] + ")"\n'
    'sys.stdout.write(d + "|" + i)\n')


def _versione_meridian():
    """(versione della distribuzione installata, versione che si riesce a
    importare). Girano in un SOTTOPROCESSO: importare meridian tira dentro
    TensorFlow e occupa la GPU, e qui la GPU deve restare libera per i fit."""
    r = subprocess.run([sys.executable, '-c', _CODICE_VERSIONE],
                       capture_output=True, text=True)
    testo = (r.stdout or '').strip()
    if '|' not in testo:
        return 'assente', 'NESSUN OUTPUT: ' + ((r.stderr or '').strip()[-160:] or '?')
    return testo.split('|', 1)


_TITOLO6 = 'Meridian ' + VERSIONE_MERIDIAN
_RIMEDIO6 = ('Esegui in una cella nuova:  !pip install -q google-meridian==' +
             VERSIONE_MERIDIAN + '   poi Runtime -> Riavvia sessione e Esegui tutto. '
             'Se fallisce ancora, riportalo in chat: senza la ' + VERSIONE_MERIDIAN +
             ' esatta il fit non e' + A + ' confrontabile con l' + A + 'osservativo.')
if not (_ok1 and _ok2 and _ok3 and _ok4 and _ok5):
    # Non installiamo niente se il run non partira' comunque: sono minuti buttati.
    _ok6 = _registra(6, _TITOLO6, False,
                     'non eseguito: prima vanno sistemati i controlli falliti qui sopra')
else:
    _dist, _imp = _versione_meridian()
    _errore_pip = ''
    if _dist != VERSIONE_MERIDIAN:
        print('      trovata "' + _dist + '": installo google-meridian==' +
              VERSIONE_MERIDIAN + ', qualche minuto senza output...')
        _ins = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q',
             'google-meridian==' + VERSIONE_MERIDIAN, 'openpyxl', 'statsmodels'],
            capture_output=True, text=True)
        if _ins.returncode != 0:
            _errore_pip = (((_ins.stderr or _ins.stdout) or '').strip()[-200:]
                           or 'senza messaggio')
        else:
            _dist, _imp = _versione_meridian()
    if _errore_pip:
        _ok6 = _registra(6, _TITOLO6, False,
                         'installazione fallita: ' + _errore_pip, _RIMEDIO6)
    elif _dist != VERSIONE_MERIDIAN:
        _ok6 = _registra(6, _TITOLO6, False,
                         'installata la versione "' + _dist + '", non la ' +
                         VERSIONE_MERIDIAN, _RIMEDIO6)
    elif _imp != VERSIONE_MERIDIAN:
        _ok6 = _registra(6, _TITOLO6, False,
                         'distribuzione ' + _dist + ' ma import KO: ' + _imp,
                         'Runtime -> Riavvia sessione, poi Esegui tutto. Se l' + A +
                         'import continua a fallire riportalo in chat con il messaggio '
                         'qui sopra: e' + A + ' quasi sempre un conflitto di numpy o '
                         'TensorFlow, e va risolto prima del ciclo.')
    else:
        _ok6 = _registra(6, _TITOLO6, True,
                         VERSIONE_MERIDIAN + ' (distribuzione e import concordi)')

# --- il riepilogo: sei righe, una per controllo ------------------------------
print()
print('=' * 72)
print('PRE-VOLO, sei controlli')
for _n, _titolo, _ok, _dettaglio in _ESITI:
    print(' ' + (SPUNTA if _ok else CROCE) + '  ' + str(_n) + '. ' +
          _titolo.ljust(22) + ' ' + _dettaglio[:150])
print('=' * 72)
if _RIMEDI:
    raise SystemExit(
        'PRE-VOLO FALLITO su ' + str(len(_RIMEDI)) + ' controllo/i su 6. Il ciclo '
        'NON parte: meglio fermarsi adesso che scoprirlo fra un' + A + 'ora e mezza.\n\n' +
        '\n\n'.join(_RIMEDI) + '\n\nSistemato il problema: Esegui tutto di nuovo. '
        'Quello che era gia' + A + ' stato fatto non si rifa' + A + '.')
print('pre-volo superato: 6 su 6.')

# ============================================================================
# SETUP - repo, patch del generatore, runner
# ============================================================================
_BLOB = (
    '...')

_dati = json.loads(base64.b64decode(''.join(_BLOB)).decode('utf-8'))

# --- repo: clona main (una cartella parziale di un clone fallito va rimossa) --
print()
print('SETUP')
print('=' * 72)
if os.path.isdir(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    print(' rimuovo la cartella parziale di un clone fallito: ' + REPO)
    shutil.rmtree(REPO)
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', URL_REPO, REPO], check=True)
else:
    print(' repo gia' + A + ' presente, non riclonato')
print(' commit: ' + subprocess.run(['git', '-C', REPO, 'log', '-1', '--oneline'],
                                   capture_output=True, text=True).stdout.strip())

# --- generatore: otto sostituzioni, sempre a partire dal file committato -----
# main non ha ne' il flag --seed-mondo, ne' il fix del controllo stagionale, ne'
# la combinazione --seed-mondo + --sanita che serve al disegno randomizzato.
#
# Si riparte OGNI VOLTA dal contenuto committato invece che dal file su disco,
# anche quando il repo e' gia' li'. Il motivo: le otto sostituzioni non sono
# indipendenti - la settima e l'ottava riscrivono pezzi che la quarta ha appena
# inserito - quindi su un file gia' patchato il frammento della quarta non
# esiste piu' e non si saprebbe distinguere "gia' fatto" da "file sbagliato".
# Ripartire dal committato rende il risultato lo stesso a ogni rilancio, che e'
# esattamente cio' che serve dopo un crash di Colab. Nel clone il contenuto
# committato e' identico byte per byte al file scaricato: verificato.
_p = subprocess.run(['git', '-C', REPO, 'show', 'HEAD:genera_dati_simulati.py'],
                    capture_output=True)
if _p.returncode != 0:
    raise SystemExit('non riesco a rileggere genera_dati_simulati.py da git: ' +
                     _p.stderr.decode('utf-8', 'replace')[-300:])
_src = _p.stdout.decode('utf-8')
_n = 0
for _vecchio, _nuovo in _dati['patches']:
    if _src.count(_vecchio) != 1:
        raise SystemExit(
            'PATCH DEL GENERATORE NON APPLICABILE alla sostituzione numero ' +
            str(_n + 1) + ' su ' + str(len(_dati['patches'])) + ': il frammento '
            'atteso si trova ' + str(_src.count(_vecchio)) + ' volte invece di 1. '
            'Vuol dire che il generatore su main e' + A + ' cambiato da quando questo '
            'notebook e' + A + ' stato preparato (magari proprio perche' + A + ' queste '
            'modifiche sono state incorporate). Non proseguo con un file a '
            'meta' + A + ', che produrrebbe dati sbagliati senza dirlo: riportalo in '
            'chat, serve un notebook aggiornato. Frammento: ' + _vecchio[:90])
    _src = _src.replace(_vecchio, _nuovo)
    _n += 1
open(GEN, 'w', encoding='utf-8').write(_src)
print(' generatore: ' + str(_n) + ' sostituzioni su ' + str(len(_dati['patches'])) +
      ', a partire dal file committato')
compile(open(GEN, encoding='utf-8').read(), GEN, 'exec')

# --- runner: riscritto per intero, gia' con --disegno, canali e draw ---------
open(RUNNER, 'w', encoding='utf-8').write(_dati['runner'])
compile(_dati['runner'], RUNNER, 'exec')
print(' runner scritto: ' + RUNNER)

# --- autotest: i flag esistono davvero, non solo nel codice sorgente ---------
def _aiuto(percorso, etichetta):
    """L'help di uno script, ma con l'errore VERO se lo script non parte. Un
    autotest che dice "flag assente" quando in realta' manca una libreria manda
    a cercare nel posto sbagliato, e qui non c'e' nessuno a cui chiedere."""
    r = subprocess.run([sys.executable, percorso, '--help'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            'AUTOTEST FALLITO: ' + etichetta + ' non parte nemmeno per stampare '
            'l' + A + 'help (codice di uscita ' + str(r.returncode) + '). Non lancio '
            'il ciclo. Errore:\n' + ((r.stderr or r.stdout) or '').strip()[-600:])
    return r.stdout


_h = _aiuto(GEN, 'il generatore')
assert '--seed-mondo' in _h, 'autotest fallito: --seed-mondo assente dal generatore'
assert '--sanita' in _h, 'autotest fallito: --sanita assente dal generatore'
_h = _aiuto(RUNNER, 'il runner')
assert '--disegno' in _h, 'autotest fallito: --disegno assente dal runner'
_testo_runner = open(RUNNER, encoding='utf-8').read()
assert 'COLONNE_CANALI' in _testo_runner, 'autotest fallito: file dei canali assente'
assert 'def salva_draw' in _testo_runner, 'autotest fallito: salvataggio dei draw assente'
print(' autotest: --seed-mondo, --sanita, --disegno, canali, draw: tutti presenti')

os.makedirs(os.path.dirname(CSV_RANDOMIZZATO), exist_ok=True)
print()
print(' CSV osservativo (letto, mai toccato): ' + CSV_OSSERVATIVO)
print(' CSV di questo run (scritto)         : ' + CSV_RANDOMIZZATO)
SETUP_COMPLETO = True
print()
print('=' * 72)
print('SETUP COMPLETO. La cella successiva parte da sola e dura 80-130 minuti.')
print('=' * 72)

In [ ]:
# ============================================================================
# CELLA 2 - IL CICLO: otto mondi randomizzati, poi il riepilogo pre-registrato
# ============================================================================
# Per ogni mondo: lo rigenera con la spesa randomizzata, lo fitta (~10 minuti su
# T4), misura intervallo esatto, PIT, canali e draw, accoda UNA riga su Drive e
# libera la GPU. La riga viene scritta SUBITO dopo il fit: se la sessione muore
# al sesto mondo, i primi cinque sono gia' salvi e non si rifanno.
#
# Il riepilogo in fondo applica la regola di lettura fissata PRIMA di eseguire,
# in PREREGISTRAZIONE_copertura_randomizzata.md: il ramo si calcola, non si
# sceglie a risultati visti.
import csv, os, statistics, subprocess, sys, time

try:
    SETUP_COMPLETO
except NameError:
    raise SystemExit(
        'Esegui prima la CELLA 1 (pre-volo e setup): questa cella usa il repo, il '
        'runner e i percorsi che quella prepara e verifica.')


def durata(secondi):
    """'1h 23m 45s', sempre nello stesso formato: i tempi di otto mondi si
    confrontano a colpo d'occhio solo se sono scritti tutti uguali."""
    s = int(secondi + 0.5)
    return (str(s // 3600) + 'h ' + format((s % 3600) // 60, '02d') + 'm ' +
            format(s % 60, '02d') + 's')


def righe_ok(percorso):
    """I semi gia' misurati con successo: si salta cio' che e' gia' fatto."""
    fatti = set()
    if os.path.exists(percorso) and os.path.getsize(percorso) > 0:
        with open(percorso, newline='', encoding='utf-8') as f:
            for r in csv.DictReader(f):
                if r.get('esito') == 'ok':
                    try:
                        fatti.add(int(r['seed_mondo']))
                    except (KeyError, TypeError, ValueError):
                        pass
    return fatti


def riga_del_mondo(percorso, seme):
    """L'ultima riga scritta PER QUESTO seme, non l'ultima del file: se il
    processo e' morto senza scrivere, deve venir fuori None e non la riga di
    un altro mondo."""
    if not (os.path.exists(percorso) and os.path.getsize(percorso) > 0):
        return None
    with open(percorso, newline='', encoding='utf-8') as f:
        righe = [r for r in csv.DictReader(f)
                 if str(r.get('seed_mondo', '')).strip() == str(seme)]
    return righe[-1] if righe else None


gia_fatti = righe_ok(CSV_RANDOMIZZATO)
da_fare = [s for s in SEMI_MONDO if s not in gia_fatti]
print('CSV di questo run: ' + CSV_RANDOMIZZATO)
print('mondi gia' + A + ' completati: ' +
      (str(sorted(gia_fatti)) if gia_fatti else 'nessuno, si parte da zero'))
print('mondi da fare    : ' + (str(da_fare) if da_fare else 'nessuno, sono tutti fatti'))

t_inizio = time.time()
durate = []
for indice, seme in enumerate(SEMI_MONDO, 1):
    if seme in gia_fatti:
        print('mondo ' + str(seme) + ': gia' + A + ' misurato, salto (ripresa).')
        continue
    # quanti mondi restano da qui in avanti, questo compreso: serve per la stima
    rimasti_prima = len(da_fare) - da_fare.index(seme) if seme in da_fare else 0
    print()
    print('=' * 72)
    print('MONDO ' + str(seme) + ' RANDOMIZZATO  (' + str(indice) + ' di ' +
          str(len(SEMI_MONDO)) + ')  - genera, fitta (~10 min), misura')
    if durate:
        print('trascorso finora: ' + durata(time.time() - t_inizio) +
              '   |   stima per ' +
              ('il mondo che resta' if rimasti_prima == 1
               else 'i ' + str(rimasti_prima) + ' mondi che restano') + ': ' +
              durata(statistics.mean(durate) * rimasti_prima))
    print('=' * 72)
    # stdin CHIUSO: una domanda imprevista diventa un errore immediato invece di
    # un blocco infinito in attesa di un click che, a run non presidiato, non
    # arriverebbe mai. A fine processo la memoria della GPU torna libera.
    t_mondo = time.time()
    esito_proc = subprocess.run(
        [sys.executable, RUNNER, '--seed-mondo', str(seme),
         '--disegno', 'randomizzato',
         '--repo', REPO, '--csv', CSV_RANDOMIZZATO, '--keep', str(KEEP)],
        stdin=subprocess.DEVNULL)
    dt = time.time() - t_mondo
    durate.append(dt)

    ultima = riga_del_mondo(CSV_RANDOMIZZATO, seme)
    if ultima is None:
        # Il processo e' morto senza scrivere: GPU persa, memoria finita, sessione
        # scollegata. Non fermiamo il ciclo, gli altri mondi possono ancora
        # riuscire; il mondo mancante si vede dal conteggio finale.
        print('[ATTENZIONE] mondo ' + str(seme) + ': nessuna riga registrata (codice '
              'di uscita ' + str(esito_proc.returncode) + '). Proseguo con i mondi '
              'successivi: questo si rifara' + A + ' rilanciando la cella.')
    else:
        print('riga registrata: ' + str(ultima.get('mondo')) +
              ' | esito=' + str(ultima.get('esito')) +
              ' | dentro=' + str(ultima.get('dentro')) +
              ' | PIT=' + str(ultima.get('pit')) +
              ' | mediana/vero=' + str(ultima.get('rapporto_mediana')) +
              (' | ERRORE: ' + str(ultima.get('errore'))[:120]
               if ultima.get('errore') else ''))
    print('tempo di questo mondo: ' + durata(dt) +
          '   |   totale trascorso: ' + durata(time.time() - t_inizio))

print()
print('ciclo finito. Tempo totale: ' + durata(time.time() - t_inizio) +
      ('   (media per mondo: ' + durata(statistics.mean(durate)) + ' su ' +
       str(len(durate)) + ' mondi fittati ora)' if durate else
       '   (nessun mondo fittato ora: erano gia' + A + ' tutti fatti)'))

# ============================================================================
# RIEPILOGO: applica la regola PRE-REGISTRATA, non la sceglie
# ============================================================================
def conteggi(pit):
    """dentro, alti (sottostima), bassi (sovrastima), sparsi. Le quattro
    quantita' del 5 della pre-registrazione, con le sue soglie."""
    return (sum(1 for p in pit if 0.05 <= p <= 0.95),
            sum(1 for p in pit if p > 0.95),
            sum(1 for p in pit if p < 0.05),
            sum(1 for p in pit if 0.20 <= p <= 0.80))


print()
print('#' * 72)
print('COPERTURA SUL REGIME RANDOMIZZATO')
print('#' * 72)
with open(CSV_RANDOMIZZATO, newline='', encoding='utf-8') as f:
    _tutte = [r for r in csv.DictReader(f)
              if r.get('esito') == 'ok' and int(r['seed_mondo']) in SEMI_MONDO]
# Un mondo rifatto dopo un crash compare due volte: vale l'ULTIMA misura. Senza
# questo, quel mondo peserebbe il doppio su copertura, PIT e mediane, e il ramo
# potrebbe uscire diverso da quello vero.
_per_seme = {}
for r in _tutte:
    _per_seme[int(r['seed_mondo'])] = r
righe = [_per_seme[s] for s in sorted(_per_seme)]
if len(_tutte) != len(righe):
    print('[nota] ' + str(len(_tutte) - len(righe)) + ' riga/e doppia/e nel CSV '
          '(mondi rifatti): ognuno conta una volta sola, con l' + A + 'ultima misura.')
N = len(righe)
if N == 0:
    raise SystemExit('Nessun mondo riuscito: niente da riassumere. Guarda gli errori '
                     'qui sopra e riportali in chat.')
pit = [float(r['pit']) for r in righe]
dentro, alti, bassi, sparsi = conteggi(pit)
mediana_pit = statistics.median(pit)

print('mondi riusciti: ' + str(N) + ' su ' + str(len(SEMI_MONDO)))
print('dentro=' + str(dentro) + '  alti(PIT>0,95, SOTTOSTIMA)=' + str(alti) +
      '  bassi(PIT<0,05, SOVRASTIMA)=' + str(bassi) +
      '  sparsi(0,20-0,80)=' + str(sparsi))
print('PIT in ordine: ' + ', '.join(format(p, '.4f') for p in sorted(pit)))
print('mediana dei PIT: ' + format(mediana_pit, '.4f'))
print('mediana/vero per mondo: ' +
      ', '.join(str(r['seed_mondo']) + ':' + str(r['rapporto_mediana']) for r in righe))

# diagnostica separata: la colonna "dentro" (ci05<=vero<=ci95) deve concordare
# con il conteggio fatto dai PIT. Se non concorda e' un caso di bordo sui
# quantili, e va guardato a mano prima di scrivere qualunque conclusione.
_dentro_col = sum(int(r['dentro']) for r in righe)
if _dentro_col != dentro:
    print('[diagnostica] dentro da PIT=' + str(dentro) + ', dalla colonna=' +
          str(_dentro_col) + ': caso di bordo, da guardare a mano.')


# soglie riscalate se N < 8, arrotondamento con 0,5 VERSO L'ALTO (fissato nel
# 5 della pre-registrazione: non dipende dalla convenzione di round())
def _soglia(x):
    return int(x * N / 8.0 + 0.5)


if dentro <= _soglia(3):
    ramo = ('1 - FERMARSI: il problema non e' + A + ' il regime, e' + A + ' il modello '
            'o i prior. NESSUNA conclusione va scritta.')
elif dentro >= _soglia(7) and sparsi >= _soglia(4):
    ramo = ('2 - CALIBRATO E CENTRATO: nel randomizzato la copertura torna e la '
            'posterior non e' + A + ' spostata.')
elif dentro >= _soglia(7):
    # La direzione NON puo' venire da alti/bassi: in questo ramo sono quasi
    # sempre entrambi zero. Viene da DOVE stanno i PIT dentro l'intervallo.
    _dir = ('SOTTOSTIMA (il vero sta in alto nella posterior)' if mediana_pit > 0.5
            else 'SOVRASTIMA (il vero sta in basso nella posterior)')
    ramo = ('3 - COPERTO MA SPOSTATO: l' + A + 'intervallo contiene il vero ma '
            'addossato a un bordo. Direzione: ' + _dir + ' (mediana dei PIT ' +
            format(mediana_pit, '.4f') + '). Mai scrivere "calibrato" senza il '
            'qualificatore.')
elif _soglia(4) <= dentro <= _soglia(6) and alti > bassi:
    ramo = ('4 - IL VERSO SI RIBALTA COL REGIME: l' + A + 'osservativo manca in '
            'basso, il randomizzato in alto.')
else:
    ramo = ('5 - COPERTURA DEGRADATA, REGIME NON SUFFICIENTE: la lettura "l' + A +
            'osservativo sovrastima perche' + A + ' e' + A + ' osservativo" NON regge '
            'da sola. Va dichiarato nella prima riga della consegna.')
print()
print('RAMO PRE-REGISTRATO: ' + ramo)
print('(regola fissata in PREREGISTRAZIONE_copertura_randomizzata.md, 5)')

# --- 5-bis: le due misure sui canali ----------------------------------------
_canali_csv = (CSV_RANDOMIZZATO[:-4] + '_canali.csv'
               if CSV_RANDOMIZZATO.lower().endswith('.csv')
               else CSV_RANDOMIZZATO + '_canali.csv')
if os.path.exists(_canali_csv):
    with open(_canali_csv, newline='', encoding='utf-8') as f:
        _ch = [r for r in csv.DictReader(f)]
    # stessa regola del CSV principale: di un mondo rifatto vale l'ultima
    # scrittura, altrimenti quel mondo avrebbe 14 righe invece di 7.
    _ultimo = {}
    for r in _ch:
        _ultimo[(r.get('seed_mondo'), r.get('canale'))] = r
    if len(_ch) != len(_ultimo):
        print('[nota] ' + str(len(_ch) - len(_ultimo)) + ' riga/e doppia/e nel file '
              'dei canali (mondi rifatti): tengo l' + A + 'ultima scrittura.')
    _per_mondo = {}
    for r in _ultimo.values():
        _per_mondo.setdefault(int(r['seed_mondo']), []).append(r)
    _dentro_per_seme = {int(r['seed_mondo']): (0.05 <= float(r['pit']) <= 0.95)
                        for r in righe}
    print()
    print('5-bis - i canali (atteso A: >=5 canali su 7 con larghezza > 1 anche '
          'nei mondi col totale coperto; atteso B: Spearman mediano < 0,5)')
    _rho = []
    for _s in sorted(_per_mondo):
        _righe_c = _per_mondo[_s]
        _larghi = sum(1 for r in _righe_c
                      if r['larghezza_relativa'] not in ('', None)
                      and float(r['larghezza_relativa']) > 1.0)
        _r = _righe_c[0].get('rho_spearman', '')
        if _r not in ('', None):
            _rho.append(float(_r))
        print('  mondo ' + str(_s).ljust(4) + ' canali con forbice > 1x la mediana: ' +
              str(_larghi) + '/7   Spearman stima-vero: ' + str(_r) +
              ('   (totale coperto)' if _dentro_per_seme.get(_s) else ''))
    if _rho:
        _med_rho = statistics.median(_rho)
        print('  Spearman mediano sugli ' + str(len(_rho)) + ' mondi: ' +
              format(_med_rho, '.4f') + '  -> ' +
              ('i canali si possono usare per ORDINARE le priorita' + A + ' di test '
               '(contraddice l' + A + 'attesa B: dichiararlo)' if _med_rho >= 0.5 else
               ('la classifica e' + A + ' INVERTITA: il totale convince e i canali '
                'ingannano' if _med_rho < 0 else
                'i canali non sono usabili nemmeno per ordinare')))
    _coperti = [s for s in _per_mondo if _dentro_per_seme.get(s)]
    if _coperti:
        _ok_a = sum(1 for s in _coperti
                    if sum(1 for r in _per_mondo[s]
                           if r['larghezza_relativa'] not in ('', None)
                           and float(r['larghezza_relativa']) > 1.0) >= 5)
        print('  attesa A verificata in ' + str(_ok_a) + ' dei ' + str(len(_coperti)) +
              ' mondi col totale coperto')
    else:
        print('  attesa A: nessun mondo col totale coperto, non calcolabile')
else:
    print()
    print('[avviso] file dei canali assente (' + _canali_csv + '): il 5-bis non '
          'e' + A + ' calcolabile. Il runner non aveva la parte dei canali?')

# --- confronto affiancato con l'osservativo ---------------------------------
# ENTRAMBI i lati calcolati con la STESSA definizione (0,05 <= PIT <= 0,95),
# quella della pre-registrazione: non la colonna "dentro" da una parte e il PIT
# dall'altra, che darebbe due numeri non confrontabili.
_pit_oss = [float(r['pit']) for r in RIGHE_OSSERVATIVE]
_dentro_oss, _alti_oss, _bassi_oss, _sparsi_oss = conteggi(_pit_oss)
print()
print('OSSERVATIVO vs RANDOMIZZATO, mondo per mondo')
print('  seme |  oss: dentro  mediana/vero     PIT |  rand: dentro  mediana/vero     PIT')
_o = {int(r['seed_mondo']): r for r in RIGHE_OSSERVATIVE}
_r2 = {int(r['seed_mondo']): r for r in righe}


def _f(x):
    if not x:
        return '        (assente)        '
    _d = 1 if 0.05 <= float(x['pit']) <= 0.95 else 0
    return '  %6s %13s %7s' % (_d, x['rapporto_mediana'],
                               format(float(x['pit']), '.4f'))


for s in sorted(SEMI_MONDO):
    print('  %4d |%s |%s' % (s, _f(_o.get(s)), _f(_r2.get(s))))
print()
print('copertura (stessa definizione, 0,05 <= PIT <= 0,95):')
print('  osservativo   ' + str(_dentro_oss) + '/' + str(len(RIGHE_OSSERVATIVE)) +
      '   (alti=' + str(_alti_oss) + ', bassi=' + str(_bassi_oss) + ')')
print('  randomizzato  ' + str(dentro) + '/' + str(N) +
      '   (alti=' + str(alti) + ', bassi=' + str(bassi) + ')')

# --- cosa scaricare e cosa riportare ----------------------------------------
print()
print('#' * 72)
print('DA FARE ADESSO')
print('#' * 72)
print('1. Scarica da Drive, cartella ' + os.path.dirname(CSV_RANDOMIZZATO) + ':')
print('   - ' + os.path.basename(CSV_RANDOMIZZATO) + '   (una riga per mondo)')
print('   - ' + os.path.basename(_canali_csv) + '   (una riga per mondo x canale)')
_dir_draw = (CSV_RANDOMIZZATO[:-4] if CSV_RANDOMIZZATO.lower().endswith('.csv')
             else CSV_RANDOMIZZATO) + '_draw'
print('   - ' + os.path.basename(_dir_draw) + '/   (i draw, un .csv.gz per mondo)')
print('2. Riporta in chat la riga "RAMO PRE-REGISTRATO" qui sopra, per intera e')
print('   come prima riga: il ramo, non i singoli PIT.')